In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('NGAsub_flatfile.csv')
print(df.shape)
df.columns.tolist()

(71340, 227)


['NGAsubRSN',
 'DatabaseRegion',
 'NGAsubEQID',
 'NGAsubSSN',
 'Earthquake_Name',
 'YEAR',
 'MODY',
 'HRMN',
 'Earthquake_Magnitude',
 'Hypocenter_Latitude_deg',
 'Hypocenter_Longitude_deg',
 'Hypocenter_Depth_km',
 'Strike_deg',
 'Dip_deg',
 'Rake_deg',
 'Ztor_km',
 'Zbor_km',
 'Intra_Inter_Flag',
 'Finite_Fault_Model',
 'Multiple_Event',
 'Number_of_Segment',
 'Length_km',
 'Width_km',
 'Source_review_flag',
 'Fault_Type',
 'Subduction_Zone_Name',
 'Alternate_Saturation_Magnitude_10MPa',
 'Preferred_Saturation_Magnitude_10MPa',
 'Alternate_Fault_maximum_width_km',
 'Preferred_Fault_maximum_width_km',
 'Magnitude_break_interface',
 'VolcanicArcFlag_Hypocenter',
 'Interface_Event_FromHypocenterDepth',
 'InterfaceFlag_FromHypocenterDepth',
 'Instrument_Type',
 'Accelerograph_Seismograph',
 'Unfiltered_PGA_H1_g',
 'Unfiltered_PGA_H2_g',
 'Unfiltered_PGA_V_g',
 'Late_P_trigger_flag_1ptYes_0ptNo',
 'Shortest_Usable_Period_for_PSa_Ave_Component_sec',
 'Longest_Usable_Period_for_PSa_Ave_Comp

In [4]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('NGAsub_flatfile.csv')

print(f"Initial records: {len(df)}")

# --------------------------------------------------
# (1) Remove bad PGA and low magnitude
# --------------------------------------------------
df = df[
    (df['Earthquake_Magnitude'] >= 4) &
    (df['PGA_g'] > 0) &
    (df['PGA_g'] <= 10)
]
print(f"After Filter 1 (Mw>=4, valid PGA): {len(df)}")

# --------------------------------------------------
# (2) Remove missing key inputs
# --------------------------------------------------
required_cols = [
    'Earthquake_Magnitude',
    'Rjb_km',
    'Ztor_km',
    'Hypocenter_Depth_km',
    'Vs30_Selected_for_Analysis_m_s'
]

df = df.dropna(subset=required_cols)
print(f"After Filter 2 (remove missing inputs): {len(df)}")

# --------------------------------------------------
# (3) Keep only subduction events
# Intra_Inter_Flag = 0,1,5
# --------------------------------------------------
df = df[df['Intra_Inter_Flag'].isin([0, 1, 5])]
print(f"After Filter 3 (subduction only): {len(df)}")

# --------------------------------------------------
# (4) Depth filtering
# Interface (0): depth < 40 km
# Intraslab (1,5): depth < 200 km
# --------------------------------------------------
df = df[
    ((df['Intra_Inter_Flag'] == 0) & (df['Hypocenter_Depth_km'] < 40)) |
    ((df['Intra_Inter_Flag'].isin([1, 5])) & (df['Hypocenter_Depth_km'] < 200))
]
print(f"After Filter 4 (depth constraints): {len(df)}")

# --------------------------------------------------
# (5) Distance filtering
# RJB <= min(Rmax, 1000)
# --------------------------------------------------
df = df[
    (df['Rjb_km'] <= df['RmaxForAnalysis_km']) &
    (df['Rjb_km'] <= 1000)
]
print(f"After Filter 5 (distance constraint): {len(df)}")

# --------------------------------------------------
# (6) Sensor depth <= 2 m
# --------------------------------------------------
df = df[df['Sensor_Depth_m'] <= 2]
print(f"After Filter 6 (sensor depth <=2m): {len(df)}")

# --------------------------------------------------
# (7) Remove multiple-source events
# Multiple_Event == 0
# --------------------------------------------------
df = df[df['Multiple_Event'] == 0]
print(f"After Filter 7 (remove multi-source): {len(df)}")

# --------------------------------------------------
# (8) Remove late P-wave trigger
# Late_P_trigger_flag_1ptYes_0ptNo == 0
# --------------------------------------------------
df = df[df['Late_P_trigger_flag_1ptYes_0ptNo'] == 0]
print(f"After Filter 8 (no late P-wave): {len(df)}")

# --------------------------------------------------
# (9) Keep free-field records
# Remove GMX starting with N, Z, F
# --------------------------------------------------
df = df[
    ~df['Geomatrix_Site_Code_1st_Letter'].astype(str).str.startswith(('N','Z','F'))
]
print(f"After Filter 9 (free-field only): {len(df)}")

# --------------------------------------------------
# (10) High-quality source review
# Source_review_flag in [0,1,2,4]
# --------------------------------------------------
df = df[df['Source_review_flag'].isin([0,1,2,4])]
print(f"After Filter 10 (high-quality sources): {len(df)}")

# --------------------------------------------------
# (11) Ensure usable period >= 10 sec
# --------------------------------------------------
df = df[df['Longest_Usable_Period_for_PSa_Ave_Component_sec'] >= 10]
print(f"After Filter 11 (usable period >=10s): {len(df)}")

# --------------------------------------------------
# (12) Remove events with < 3 records
# NGAsubEQID = event ID
# --------------------------------------------------
event_counts = df['NGAsubEQID'].value_counts()
valid_events = event_counts[event_counts >= 3].index

df = df[df['NGAsubEQID'].isin(valid_events)]
print(f"After Filter 12 (>=3 records per event): {len(df)}")

# --------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------
print("\nFinal dataset summary:")
print(f"Total records: {len(df)}")
print(f"Total events: {df['NGAsubEQID'].nunique()}")
print(f"Total stations: {df['NGAsubSSN'].nunique()}")

Initial records: 71340
After Filter 1 (Mw>=4, valid PGA): 68375
After Filter 2 (remove missing inputs): 68375
After Filter 3 (subduction only): 45719
After Filter 4 (depth constraints): 38032
After Filter 5 (distance constraint): 20088
After Filter 6 (sensor depth <=2m): 19572
After Filter 7 (remove multi-source): 17764
After Filter 8 (no late P-wave): 6770
After Filter 9 (free-field only): 6770
After Filter 10 (high-quality sources): 6770
After Filter 11 (usable period >=10s): 2062
After Filter 12 (>=3 records per event): 1981

Final dataset summary:
Total records: 1981
Total events: 79
Total stations: 1037


In [5]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('NGAsub_flatfile.csv')

print(f"Initial records: {len(df)}")

# --------------------------------------------------
# (1) Remove bad PGA and low magnitude
# --------------------------------------------------
df = df[
    (df['Earthquake_Magnitude'] >= 4) &
    (df['PGA_g'] > 0) &
    (df['PGA_g'] <= 10)
]
print(f"After Filter 1 (Mw>=4, valid PGA): {len(df)}")

# --------------------------------------------------
# (2) Remove missing key inputs
# --------------------------------------------------
required_cols = [
    'Earthquake_Magnitude',
    'Rjb_km',
    'Ztor_km',
    'Hypocenter_Depth_km',
    'Vs30_Selected_for_Analysis_m_s'
]
df = df.dropna(subset=required_cols)
# Also remove -999 sentinel values in these columns
for col in required_cols:
    df = df[df[col] != -999]
print(f"After Filter 2 (remove missing inputs): {len(df)}")

# --------------------------------------------------
# (3) Keep only subduction events
# Intra_Inter_Flag = 0 (interface), 1 (intraslab), 5 (lower DSZ Japan)
# --------------------------------------------------
df = df[df['Intra_Inter_Flag'].isin([0, 1, 5])]
print(f"After Filter 3 (subduction only): {len(df)}")

# --------------------------------------------------
# (4) Depth filtering
# Interface (0): depth < 40 km
# Intraslab (1, 5): depth < 200 km
# --------------------------------------------------
df = df[
    ((df['Intra_Inter_Flag'] == 0) & (df['Hypocenter_Depth_km'] < 40)) |
    ((df['Intra_Inter_Flag'].isin([1, 5])) & (df['Hypocenter_Depth_km'] < 200))
]
print(f"After Filter 4 (depth constraints): {len(df)}")

# --------------------------------------------------
# (5) Distance filtering: RJB <= min(Rmax, 1000 km)
# --------------------------------------------------
df = df[
    (df['Rjb_km'] <= df['RmaxForAnalysis_km']) &
    (df['Rjb_km'] <= 1000)
]
print(f"After Filter 5 (distance constraint): {len(df)}")

# --------------------------------------------------
# (6) Sensor depth <= 2 m
# --------------------------------------------------
df = df[
    (df['Sensor_Depth_m'] >= 0) &   # exclude -999 sentinel
    (df['Sensor_Depth_m'] <= 2)
]
print(f"After Filter 6 (sensor depth <=2m): {len(df)}")

# --------------------------------------------------
# (7) Remove multiple-source events
# FIX: use != 1 (not == 0) so that -999 (N/A) records are retained
# --------------------------------------------------
df = df[df['Multiple_Event'] != 1]
print(f"After Filter 7 (remove multi-source): {len(df)}")

# --------------------------------------------------
# (8) Remove late P-wave trigger records
# FIX: use != 1 (not == 0) so that -999 (N/A) records are retained
# Column name: Late_P_trigger_flag_1ptYes_0ptNo
# --------------------------------------------------
df = df[df['Late_P_trigger_flag_1ptYes_0ptNo'] != 1]
print(f"After Filter 8 (no late P-wave): {len(df)}")

# --------------------------------------------------
# (9) Keep free-field records only
# Remove GMX first letter = N (structure), Z (borehole), F (dam/foundation)
# --------------------------------------------------
gmx = df['Geomatrix_Site_Code_1st_Letter'].astype(str).str.strip().str.upper()
df = df[~gmx.str.startswith(('N', 'Z', 'F'))]
print(f"After Filter 9 (free-field only): {len(df)}")

# --------------------------------------------------
# (10) High-quality source review flag = 0, 1, 2, or 4
# --------------------------------------------------
df = df[df['Source_review_flag'].isin([0, 1, 2, 4])]
print(f"After Filter 10 (high-quality sources): {len(df)}")

# --------------------------------------------------
# (11) Ensure longest usable period >= 10 sec
# Exclude -999 sentinel (missing/inapplicable)
# --------------------------------------------------
df = df[df['Longest_Usable_Period_for_PSa_Ave_Component_sec'] >= 10]
print(f"After Filter 11 (usable period >=10s): {len(df)}")

# --------------------------------------------------
# (12) Remove events with fewer than 3 records
# --------------------------------------------------
event_counts = df['NGAsubEQID'].value_counts()
valid_events = event_counts[event_counts >= 3].index
df = df[df['NGAsubEQID'].isin(valid_events)]
print(f"After Filter 12 (>=3 records per event): {len(df)}")

# --------------------------------------------------
# FINAL SUMMARY (target: ~9009 records, 189 events, 2877 stations)
# --------------------------------------------------
print("\nFinal dataset summary:")
print(f"Total records:  {len(df)}")
print(f"Total events:   {df['NGAsubEQID'].nunique()}")
print(f"Total stations: {df['NGAsubSSN'].nunique()}")

Initial records: 71340
After Filter 1 (Mw>=4, valid PGA): 68375
After Filter 2 (remove missing inputs): 65100
After Filter 3 (subduction only): 45383
After Filter 4 (depth constraints): 37749
After Filter 5 (distance constraint): 19902
After Filter 6 (sensor depth <=2m): 4092
After Filter 7 (remove multi-source): 4090
After Filter 8 (no late P-wave): 4087
After Filter 9 (free-field only): 4087
After Filter 10 (high-quality sources): 4081
After Filter 11 (usable period >=10s): 2699
After Filter 12 (>=3 records per event): 2597

Final dataset summary:
Total records:  2597
Total events:   147
Total stations: 861


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('NGAsub_flatfile.csv')
print(f"Initial records: {len(df)}")

# (1) Remove bad PGA and low magnitude
df = df[
    (df['Earthquake_Magnitude'] >= 4) &
    (df['PGA_g'] > 0) &
    (df['PGA_g'] <= 10)
]
print(f"After Filter 1 (Mw>=4, valid PGA): {len(df)}")

# (2) Remove missing key inputs — dropna only, no -999 exclusion
required_cols = [
    'Earthquake_Magnitude', 'Rjb_km', 'Ztor_km',
    'Hypocenter_Depth_km', 'Vs30_Selected_for_Analysis_m_s'
]
df = df.dropna(subset=required_cols)
print(f"After Filter 2 (remove missing inputs): {len(df)}")

# (3) Keep only subduction events (interface=0, intraslab=1,5)
df = df[df['Intra_Inter_Flag'].isin([0, 1, 5])]
print(f"After Filter 3 (subduction only): {len(df)}")

# (4) Depth filtering
df = df[
    ((df['Intra_Inter_Flag'] == 0) & (df['Hypocenter_Depth_km'] < 40)) |
    ((df['Intra_Inter_Flag'].isin([1, 5])) & (df['Hypocenter_Depth_km'] < 200))
]
print(f"After Filter 4 (depth constraints): {len(df)}")

# (5) Distance filtering: RJB <= min(Rmax, 1000 km)
df = df[
    (df['Rjb_km'] <= df['RmaxForAnalysis_km']) &
    (df['Rjb_km'] <= 1000)
]
print(f"After Filter 5 (distance constraint): {len(df)}")

# (6) Sensor depth <= 2 m
# FIX: Only EXCLUDE records explicitly known to be > 2m.
# Keep records where sensor depth is -999 (unknown/not recorded) or NaN.
df = df[
    df['Sensor_Depth_m'].isna() |           # NaN → unknown, keep
    (df['Sensor_Depth_m'] == -999) |        # sentinel → unknown, keep
    (df['Sensor_Depth_m'] <= 2)             # known and acceptable, keep
]
print(f"After Filter 6 (sensor depth <=2m): {len(df)}")

# (7) Remove multiple-source events (keep unknown -999 and 0)
df = df[df['Multiple_Event'] != 1]
print(f"After Filter 7 (remove multi-source): {len(df)}")

# (8) Remove late P-wave trigger (keep unknown -999 and 0)
df = df[df['Late_P_trigger_flag_1ptYes_0ptNo'] != 1]
print(f"After Filter 8 (no late P-wave): {len(df)}")

# (9) Keep free-field records only (exclude GMX first letter N, Z, F)
gmx = df['Geomatrix_Site_Code_1st_Letter'].astype(str).str.strip().str.upper()
df = df[~gmx.str.startswith(('N', 'Z', 'F'))]
print(f"After Filter 9 (free-field only): {len(df)}")

# (10) High-quality source review flag = 0, 1, 2, or 4
df = df[df['Source_review_flag'].isin([0, 1, 2, 4])]
print(f"After Filter 10 (high-quality sources): {len(df)}")

# (11) Longest usable period >= 10 sec
df = df[df['Longest_Usable_Period_for_PSa_Ave_Component_sec'] >= 10]
print(f"After Filter 11 (usable period >=10s): {len(df)}")

# (12) Remove events with fewer than 3 records
event_counts = df['NGAsubEQID'].value_counts()
valid_events = event_counts[event_counts >= 3].index
df = df[df['NGAsubEQID'].isin(valid_events)]
print(f"After Filter 12 (>=3 records per event): {len(df)}")

print("\nFinal dataset summary:")
print(f"Total records:  {len(df)}")
print(f"Total events:   {df['NGAsubEQID'].nunique()}")
print(f"Total stations: {df['NGAsubSSN'].nunique()}")

Initial records: 71340
After Filter 1 (Mw>=4, valid PGA): 68375
After Filter 2 (remove missing inputs): 68375
After Filter 3 (subduction only): 45719
After Filter 4 (depth constraints): 38032
After Filter 5 (distance constraint): 20088
After Filter 6 (sensor depth <=2m): 19572
After Filter 7 (remove multi-source): 18976
After Filter 8 (no late P-wave): 18351
After Filter 9 (free-field only): 18351
After Filter 10 (high-quality sources): 17126
After Filter 11 (usable period >=10s): 9074
After Filter 12 (>=3 records per event): 8963

Final dataset summary:
Total records:  8963
Total events:   188
Total stations: 2927


In [7]:
# count values -999
print("Count of -999 values:")
for col in required_cols:
    count_neg999 = (df[col] == -999).sum()
    print(f"  {col}: {count_neg999}")
    

Count of -999 values:
  Earthquake_Magnitude: 0
  Rjb_km: 22
  Ztor_km: 0
  Hypocenter_Depth_km: 0
  Vs30_Selected_for_Analysis_m_s: 53


In [8]:
# (2) Remove missing key inputs — dropna + explicit -999 sentinel removal
required_cols = [
    'Earthquake_Magnitude', 'Rjb_km', 'Ztor_km',
    'Hypocenter_Depth_km', 'Vs30_Selected_for_Analysis_m_s'
]
df = df.dropna(subset=required_cols)

# Drop -999 sentinels only where they actually exist (Rjb and Vs30)
df = df[df['Rjb_km'] != -999]
df = df[df['Vs30_Selected_for_Analysis_m_s'] != -999]

print(f"After Filter 2 (remove missing inputs): {len(df)}")

After Filter 2 (remove missing inputs): 8907


In [9]:
# save as filtered csv
df.to_csv('NGAsub_flatfile_filtered.csv', index=False)